In [ ]:
from recmodels.recmodels.sequential.gru4rec.gru4rec_hidasi import GRU4Rec
import pickle 

In [2]:
params = {
    "loss": "bpr-max",
    "n_epochs": 1
}

In [3]:
model = GRU4Rec(**params)

In [4]:
SIMULATION_PATH = "/home/caio/dev/dynamicTasteDistortion/dynamicTasteDistortion/data/simulation"

def load_bootstrapped_clicks_ipynb(data_type, file_size, num_users):
    output_path = (
    f"{SIMULATION_PATH}/{data_type}_{file_size}_n_users={num_users}_bootstrapped.pkl"
)
    with open(output_path, "rb") as f:
        bootstrapped_clicks = pickle.load(f)
    return bootstrapped_clicks


In [5]:
bootstrapped_yelp = load_bootstrapped_clicks_ipynb("yelp", "1m", 1000)

In [6]:
df = bootstrapped_yelp

In [7]:
df = df.sort_values(["user", "timestamp"], kind="stable")
chunk = df.groupby("user").cumcount() // 50
df["sessionId"] = df.groupby(["user", chunk]).ngroup()
df = df[df.groupby("sessionId")["item"].transform("size") >= 2]

In [8]:
model.fit(df, item_key="item", session_key="sessionId", time_key="timestamp")

The dataframe is already sorted by sessionId, timestamp
Created sample store with 4882 batches of samples (type=GPU)
Epoch1 --> loss: 0.692401 	(8.85s) 	[894.40 mb/s | 57242 e/s]


In [9]:
df

,user,item,relevant,clicked_at,timestamp,rating,constant,linear_time,genres,sessionId
9,0,2620,1.0,21,1505.540287,5.885868e-01,1,0.000000e+00,"(lounges, cocktail-bars, american-(new), resta...",0
17,0,425,1.0,70,2052.946564,7.675010e-01,1,1.645857e-04,"(grocery, food)",0
6,0,1708,1.0,10,2729.110595,5.227077e-01,1,2.505481e-04,"(oil-change-stations, automotive, tires, auto-...",0
10,0,1832,1.0,39,2871.482025,6.306614e-01,1,3.374673e-04,"(lounges, cocktail-bars, nightlife, event-plan...",0
18,0,1501,1.0,71,3322.196119,2.725736e-01,1,1.939812e-04,"(food, specialty-food, meat-shops)",0
...,...,...,...,...,...,...,...,...,...,...
515853,999,1469,1.0,57,187192.517909,5.058380e-01,1,5.019321e-01,"(arts-&-entertainment, cinema)",10792
515844,999,387,1.0,25,187647.005224,9.173746e-01,1,9.125017e-01,"(tacos, restaurants, mexican)",10792
515856,999,853,1.0,90,187959.839581,1.000000e-08,1,9.963470e-09,"(local-flavor,)",10793
515854,999,1057,1.0,63,188426.103212,3.556637e-01,1,3.552437e-01,"(restaurants, breakfast-&-brunch, american-(tr...",10793


In [23]:
import numpy as np
import torch


def map_to_model_idx(model, items):
    imap = model.data_iterator.itemidmap
    # reindex is just a indexing that handles unknown keys and returns NaN for them.
    mapped_indices = [imap.reindex(item).dropna().astype(np.int64).values for item in items]
    return mapped_indices

@torch.no_grad()
def predict(model, histories, item_col="item"):
    """Full-catalog scores for each user, via left-padded batched forward passes.

    model:     trained GRU4Rec object
    histories: DataFrame, one row per user, `item_col` = list of raw item ids (chronological)
    returns:   tensor (n_users, n_items); row r = histories row r, column i = internal item index i
    """
    dev = model.device
    sequences = histories[item_col]
    # raw ids -> internal indices, unknown items dropped
    mapped_sequences = map_to_model_idx(model, sequences)
    num_items = len(mapped_sequences)
    max_sequence_size = max((len(s) for s in mapped_sequences), default=0)
    bias = model.model.By.weight[:, 0]                      # fallback scores (empty history)
    if max_sequence_size == 0:
        # If no history is found, we just use the bias term as no hidden state is computed
        return bias.expand(num_items, -1).clone()

    # left padding: pad with 0s. because this is a real item id, we use a padding to help
    items = torch.zeros((num_items, max_sequence_size), dtype=torch.long)
    # padding mask: essentially used to understand which predictions to ignore.
    pad = torch.ones((num_items, max_sequence_size), dtype=torch.bool)
    for row_number, sequence in enumerate(mapped_sequences):
        # skips empty sequences
        if len(sequence):
            # Pastes the sequence to the len(sequence) rightmost entries of items tensor
            items[row_number, max_sequence_size - len(sequence):] = torch.tensor(sequence)
            pad[row_number, max_sequence_size - len(sequence):] = False
    items, pad = items.to(dev), pad.to(dev)

    # Initialize hidden state
    hidden_state = [torch.zeros((num_items, layer_size), device=dev) for layer_size in model.layers]
    dummy_item = torch.zeros(1, dtype=torch.long, device=dev) 

    for step in range(max_sequence_size):
        if step == max_sequence_size - 1:
            Y = None # Scores for the entire catalog
        else:
            Y = dummy_item # scores the prediction for a dummy item. We only 
            # use the output of the model in the last step, so it doesnt matter what 
            # we're scoring before.
        scores = model.model.forward(items[:, step], hidden_state, Y, training=False)
        for h in hidden_state:
            h[pad[:, step]] = 0                                # reset rows still in padding

    scores[pad[:, -1]] = bias                                    # users with empty history
    return scores

In [24]:
teste = df.groupby("sessionId").agg({"item": list})

In [28]:
teste

,item
sessionId,
0,"[2620, 425, 1708, 1832, 1501, 3301, 1684, 852,..."
1,"[2875, 2230, 3266, 2286, 2775, 1679, 912, 2021..."
2,"[130, 2143, 2670, 758, 3517, 3390, 3072, 3018,..."
3,"[2060, 3086, 3309, 2442, 3044, 3468, 1364, 162..."
4,"[1872, 496, 3142, 1394, 2544, 2586, 3381, 2679..."
...,...
10789,"[2253, 2523, 1615, 2975, 2105, 2106, 455, 658,..."
10790,"[3447, 207, 942, 2529, 646, 1372, 548, 393, 12..."
10791,"[1492, 3214, 2209, 2443, 2987, 2915, 2863, 330..."


In [25]:
result = predict(model, teste)

In [27]:
result.shape

torch.Size([10769, 3508])

In [ ]:
result.shape

torch.Size([10769, 3508])